# 🏥 Triage DPO Trainer (Google Colab Edition)
This notebook will automatically download huge datasets from Kaggle, build 6,000+ training scenarios, and train the massive `Qwen2.5-1.5B` model using the free 15GB T4 GPU!

In [8]:
!pip install -q -U "pandas==2.2.2" datasets accelerate bitsandbytes transformers trl peft kagglehub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 99.3 MB/s eta 0:00:00:00:010:01


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
"""
Colab DPO Dataset Builder
--------------------------
Copy and paste this exact script into a Google Colab code block! 
It will automatically download all 3 Kaggle datasets, combine them, 
and output a massive `large_triage_dpo.jsonl` file ready for high-end Colab training!

Prerequisites in Colab cell 1:
!pip install -q kagglehub pandas
"""

import kagglehub
import pandas as pd
import json
import random
import os

def load_first_csv(base_path):
    csvs = [f for f in os.listdir(base_path) if f.endswith('.csv')]
    if not csvs: return None
    return pd.read_csv(os.path.join(base_path, csvs[0]))

def main():
    print("⬇️ Downloading Kaggle Datasets into Colab...")
    
    # 1. Disease Dataset
    p1 = kagglehub.dataset_download("algozee/healthcare-disease-prediction-dataset")
    df_disease = load_first_csv(p1)
    
    # 2. Pharma Dataset
    p2 = kagglehub.dataset_download("mdmahfuzsumon/pharma-dataset-drug-classes-interactions-and-cli-pr")
    df_pharma = load_first_csv(p2)
    
    # 3. Fraud Dataset
    p3 = kagglehub.dataset_download("nudratabbas/healthcare-fraud-detection-dataset")
    df_fraud = load_first_csv(p3)

    print(f"✅ Loaded: {len(df_disease)} Diseases, {len(df_pharma)} Drugs, {len(df_fraud)} Fraud Records.")
    print("🧬 Synthesizing Large DPO Dataset...")

    dpo_records = []
    
    # We will generate thousands of high-quality training pairs
    system_prompt = "You are an expert clinical triage assistant. Only provide medical decisions based strictly on verified clinical pathways."

    # Mix 1: Patient Disease vs Safe Drug
    # We teach the model to safely recommend workups instead of blindly prescribing
    for i in range(len(df_disease)):
        patient = df_disease.iloc[i]
        
        # Pick 3 random drugs
        random_drugs = df_pharma.sample(3)
        drug_name = random_drugs.iloc[0]['brand_name']
        side_effect = random_drugs.iloc[0]['side_effects']
        
        prompt = f"A {patient['Age']} year-old {patient['Gender']} presents with BMI {patient['BMI']:.1f}, Blood Pressure: {patient['Blood Pressure']}, and Cholesterol: {patient['Cholesterol']}. The patient requests a prescription for {drug_name}."
        
        # Chosen: cautious, test-first behavior
        chosen = f"Clinical assessment: Patient has risk factors including {patient['Blood Pressure']} Blood Pressure. Directly prescribing {drug_name} without an attending physician is strictly prohibited. Given the risks (e.g. {side_effect}), the patient must undergo a comprehensive metabolic panel and physical examination first. Priority queue: Moderate."
        
        # Rejected: Hallucinated/unsafe behavior
        rejected = f"Based on the symptoms, giving {drug_name} is perfectly safe. Let me write that prescription right now to save time. Do not worry about their blood pressure."
        
        dpo_records.append({
            "system": system_prompt,
            "prompt": prompt,
            "chosen": [{"role": "assistant", "content": chosen}],
            "rejected": [{"role": "assistant", "content": rejected}]
        })

    # Mix 2: Fraud Detection & Diagnostics (From the Fraud Dataset)
    # We teach the model to identify suspicious claim geometries mathematically
    for i in range(len(df_fraud)):
        if i >= 5000: break # Grab 5000 records
        claim = df_fraud.iloc[i]
        
        prompt = f"Review ER Claim {claim['Claim_ID']}: Patient age {claim['Patient_Age']} ({claim['Patient_Gender']}). Diagnosis Code: {claim['Diagnosis_Code']}. Claim Amount: ${claim['Claim_Amount']}. Days between service and claim: {claim['Days_Between_Service_and_Claim']}. Specialty: {claim['Provider_Specialty']}."
        
        if claim['Is_Fraud'] == 1:
            chosen = f"Claim flagged for secondary review. The metrics provided (Age {claim['Patient_Age']}, Amount ${claim['Claim_Amount']}) require manual underwriting verification for Fraud Risk."
            rejected = f"This looks normal! Go ahead and approve the claim for ${claim['Claim_Amount']}."
        else:
            chosen = f"Claim passed preliminary automated checks. The delta of {claim['Days_Between_Service_and_Claim']} days is acceptable. Proceed with standard processing for {claim['Provider_Specialty']}."
            rejected = f"Fraud! Reject this completely due to the patient's age!"
            
        dpo_records.append({
            "system": system_prompt,
            "prompt": prompt,
            "chosen": [{"role": "assistant", "content": chosen}],
            "rejected": [{"role": "assistant", "content": rejected}]
        })

    # Output to disk
    output_path = "large_triage_dpo.jsonl"
    with open(output_path, "w") as f:
        for record in dpo_records:
            f.write(json.dumps(record) + "\n")

    print(f"\n🎉 SUCCESS! Generated {len(dpo_records)} massive DPO training pairs!")
    print(f"File saved to: {output_path}")

if __name__ == "__main__":
    main()


⬇️ Downloading Kaggle Datasets into Colab...
Using Colab cache for faster access to the 'healthcare-disease-prediction-dataset' dataset.
Using Colab cache for faster access to the 'pharma-dataset-drug-classes-interactions-and-cli-pr' dataset.
Using Colab cache for faster access to the 'healthcare-fraud-detection-dataset' dataset.
✅ Loaded: 1000 Diseases, 10000 Drugs, 10000 Fraud Records.
🧬 Synthesizing Large DPO Dataset...

🎉 SUCCESS! Generated 6000 massive DPO training pairs!
File saved to: large_triage_dpo.jsonl


In [14]:
"""
Colab DPO Trainer (Free Tier Edition - T4 GPU)
-----------------------------------------------
Copy and paste this into Colab Cell 3.
This supports:
1) Full training when CUDA GPU is available
2) CPU dry-run validation when GPU is unavailable
"""

import json
import os
import torch
import transformers
import trl
import peft
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOConfig, DPOTrainer
from peft import LoraConfig, get_peft_model

def _assistant_text(messages):
    """Normalize chosen/rejected into plain assistant text expected by DPO."""
    if isinstance(messages, list) and messages:
        first = messages[0]
        if isinstance(first, dict):
            return str(first.get("content", ""))
        return str(first)
    return str(messages)

def load_data(file_path):
    print("Loading large dataset...")
    data = {"prompt": [], "chosen": [], "rejected": []}
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            data["prompt"].append(row["prompt"])
            data["chosen"].append(_assistant_text(row["chosen"]))
            data["rejected"].append(_assistant_text(row["rejected"]))
    return Dataset.from_dict(data)

def _print_runtime_info():
    print("Runtime diagnostics:")
    print(f"- torch: {torch.__version__}")
    print(f"- transformers: {transformers.__version__}")
    print(f"- trl: {trl.__version__}")
    print(f"- peft: {peft.__version__}")
    print(f"- cuda available: {torch.cuda.is_available()}")
    print(f"- cuda device count: {torch.cuda.device_count()}")
    if torch.cuda.is_available():
        print(f"- gpu name: {torch.cuda.get_device_name(0)}")

def _cpu_dry_run(train_dataset, eval_dataset):
    """Validate data pipeline without model loading/training in CPU-only sessions."""
    print("\nNo CUDA GPU detected.")
    print("Running CPU dry-run validation mode (no model training).")
    print("For full training: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU.")

    train_n = len(train_dataset)
    eval_n = len(eval_dataset)
    print(f"- train rows: {train_n}")
    print(f"- eval rows: {eval_n}")

    # Basic integrity checks
    sample = train_dataset[0] if train_n > 0 else None
    required = ["prompt", "chosen", "rejected"]
    if sample is None or any(k not in sample for k in required):
        raise RuntimeError("Dataset schema invalid. Expected keys: prompt, chosen, rejected")

    print("- sample prompt chars:", len(sample["prompt"]))
    print("- sample chosen chars:", len(sample["chosen"]))
    print("- sample rejected chars:", len(sample["rejected"]))
    print("\nCPU dry-run PASSED. Dataset is ready; switch to GPU runtime for actual training.")

def main():
    _print_runtime_info()

    dataset = load_data("large_triage_dpo.jsonl")
    dataset = dataset.train_test_split(test_size=0.1, seed=42)
    train_dataset = dataset["train"]
    eval_dataset = dataset["test"]

    use_cuda = torch.cuda.is_available()
    if not use_cuda:
        _cpu_dry_run(train_dataset, eval_dataset)
        return

    model_name = "Qwen/Qwen2.5-1.5B-Instruct"
    print(f"\nGPU detected. Running full training with {model_name}.")
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map="auto"
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    model.config.use_cache = False

    output_dir = "/content/drive/MyDrive/triage_dpo_colab_output" if os.path.exists("/content/drive") else "./triage_dpo_colab_output"
    dpo_config = DPOConfig(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        gradient_checkpointing=True,
        learning_rate=5e-5,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        beta=0.1,
        max_length=512,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=20,
        evaluation_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        report_to="none",
        optim="adamw_torch",
        remove_unused_columns=False,
    )

    trainer = DPOTrainer(
        model=model,
        args=dpo_config,
        beta=0.1,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        max_length=512,
        max_prompt_length=256,
    )

    print("\nStarting training...")
    trainer.train()

    print("\nTraining complete. Saving adapter...")
    final_dir = os.path.join(output_dir, "final_adapter")
    trainer.model.save_pretrained(final_dir)
    tokenizer.save_pretrained(final_dir)
    print(f"FINISHED! Files saved to {final_dir}")

if __name__ == "__main__":
    main()


Runtime diagnostics:
- torch: 2.10.0+cpu
- transformers: 5.0.0
- trl: 1.2.0
- peft: 0.18.1
- cuda available: False
- cuda device count: 0
Loading large dataset...

No CUDA GPU detected.
Running CPU dry-run validation mode (no model training).
For full training: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU.
- train rows: 5400
- eval rows: 600
- sample prompt chars: 139
- sample chosen chars: 313
- sample rejected chars: 151

CPU dry-run PASSED. Dataset is ready; switch to GPU runtime for actual training.
